Copyright 2021 DeepMind Technologies Limited

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

This colab showcases training of the Enformer model published in

**"Effective gene expression prediction from sequence by integrating long-range interactions"**

Žiga Avsec, Vikram Agarwal, Daniel Visentin, Joseph R. Ledsam, Agnieszka Grabska-Barwinska, Kyle R. Taylor, Yannis Assael, John Jumper, Pushmeet Kohli, David R. Kelley


## Steps

- Setup tf.data.Dataset by directly accessing the Basenji2 data on GCS: `gs://basenji_barnyard/data`
- Train the model for a few steps, alternating training on human and mouse data batches
- Evaluate the model on human and mouse genomes

## Setup

**Start the colab kernel with GPU**: Runtime -> Change runtime type -> GPU

### Install dependencies

In [1]:
!pip install dm-sonnet tqdm

In [2]:
# Get enformer source code
!wget -q https://raw.githubusercontent.com/deepmind/deepmind-research/master/enformer/attention_module.py
!wget -q https://raw.githubusercontent.com/deepmind/deepmind-research/master/enformer/enformer.py

### Import

In [1]:
from dotenv import load_dotenv
import os
load_dotenv() # This looks for the .env file and loads the variables

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')
os.environ['GCS_PAYER_PROJECT'] = os.getenv('GCP_PROJECT_ID')

import tensorflow as tf
# Make sure the GPU is enabled 
assert tf.config.list_physical_devices('GPU'), 'Start the colab kernel with GPU: Runtime -> Change runtime type -> GPU'

tf.config.experimental_connect_to_cluster = None # Reset if needed
# Easier debugging of OOM
%env TF_ENABLE_GPU_GARBAGE_COLLECTION=false


2026-01-03 18:44:59.502719: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0


env: TF_ENABLE_GPU_GARBAGE_COLLECTION=false


2026-01-03 18:45:01.353089: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcuda.so.1
2026-01-03 18:45:01.412902: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:923] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-03 18:45:01.412945: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1733] Found device 0 with properties: 
pciBusID: 0000:01:00.0 name: NVIDIA RTX 4500 Ada Generation computeCapability: 8.9
coreClock: 2.58GHz coreCount: 60 deviceMemorySize: 23.99GiB deviceMemoryBandwidth: 402.38GiB/s
2026-01-03 18:45:01.412958: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0
2026-01-03 18:45:01.424775: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcublas.so.11
2026-01-03 18:45:01.424827: I tensorflow/stream_

In [2]:
import sonnet as snt
from tqdm import tqdm
from IPython.display import clear_output
import numpy as np
import pandas as pd
import time
import glob
import json
import functools
from google.cloud import storage


An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'


/usr/local/lib/python3.8/dist-packages/google/api_core/_python_version_support.py:237: FutureWarning: You are using a non-supported Python version (3.8.10). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)


In [3]:
assert snt.__version__.startswith('2.0')

In [4]:
tf.__version__

'2.5.0'

In [5]:
# GPU colab has T4 with 16 GiB of memory
!nvidia-smi

Sat Jan  3 18:45:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.02              Driver Version: 552.55         CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 4500 Ada Gene...    On  |   00000000:01:00.0 Off |                  Off |
| 30%   28C    P8              1W /  210W |    1365MiB /  24570MiB |      2%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Code

In [6]:
import enformer

In [7]:
# @title `get_targets(organism)`
def get_targets(organism):
  """
  Get the targets file for an organism.
  Targets files list the output tracks for each organism, like CAGE peaks or ChIP-seq signals.
  """
  targets_txt = f'https://raw.githubusercontent.com/calico/basenji/master/manuscripts/cross2020/targets_{organism}.txt'
  return pd.read_csv(targets_txt, sep='\t')

In [ ]:

# @title `get_dataset(organism, subset, num_threads=8)`
def organism_path(organism):
  return os.path.join('gs://basenji_barnyard/data', organism)


def get_dataset(organism, subset, num_threads=8):
  metadata = get_metadata(organism) # Fetch metadata
  dataset = tf.data.TFRecordDataset(tfrecord_files_local(organism, subset), # Read TFRecords
                                    compression_type='ZLIB',
                                    num_parallel_reads=num_threads)
  dataset = dataset.map(functools.partial(deserialize, metadata=metadata), # Parse TFRecords
                        num_parallel_calls=num_threads)
  return dataset


# The original function is commented out below since it does not work.
# def get_metadata(organism):
#   """
#   Get the metadata for an organism.
#   """
#   # Keys:
#   # num_targets, train_seqs, valid_seqs, test_seqs, seq_length,
#   # pool_width, crop_bp, target_length
#   path = os.path.join(organism_path(organism), 'statistics.json')
#   with tf.io.gfile.GFile(path, 'r') as f:
#     return json.load(f)

def get_metadata(organism):
    """
    Get the metadata for an organism.
    """
    client = storage.Client(project=os.getenv('GCP_PROJECT_ID'))
    
    # Parse the GCS URL
    bucket_name = 'basenji_barnyard'
    blob_path = f'data/{organism}/statistics.json'
    
    bucket = client.bucket(bucket_name, user_project=os.getenv('GCP_PROJECT_ID'))
    blob = bucket.blob(blob_path)
    
    # Download as text
    content = blob.download_as_text()
    return json.loads(content)

# This original function has the same issue. 
# def tfrecord_files(organism, subset):
#   """
#   Get sorted list of TFRecord files for an organism and subset.
#   """
#   # Sort the values by int(*).
#   return sorted(tf.io.gfile.glob(os.path.join(
#       organism_path(organism), 'tfrecords', f'{subset}-*.tfr'
#   )), key=lambda x: int(x.split('-')[-1].split('.')[0]))
def tfrecord_files_gcs(organism, subset):
    """
    Get sorted list of TFRecord files for an organism and subset using GCS client.
    """
    client = storage.Client(project=os.getenv('GCP_PROJECT_ID'))
    bucket_name = 'basenji_barnyard'
    
    # The directory prefix where the tfrecords are stored
    prefix = f'data/{organism}/tfrecords/{subset}-'
    
    # Get the bucket
    bucket = client.bucket(bucket_name, user_project=os.getenv('GCP_PROJECT_ID'))
    
    # List all blobs that start with the specific subset prefix
    blobs = bucket.list_blobs(prefix=prefix)
    
    # Create the full GCS paths (gs://bucket_name/blob_name)
    # and filter to ensure they end with .tfr
    files = [
        f"gs://{bucket_name}/{blob.name}" 
        for blob in blobs 
        if blob.name.endswith('.tfr')
    ]
    
    # Sort the values by the integer index at the end: {subset}-{index}.tfr
    return sorted(files, key=lambda x: int(x.split('-')[-1].split('.')[0]))

def tfrecord_files_local(organism, subset, base_path='data'):
    """
    Get sorted list of TFRecord files for an organism and subset from local disk.
    """
    # Construct the search pattern (equivalent to the GCS prefix)
    # This assumes a structure like: data/human/tfrecords/train-*.tfr
    search_pattern = os.path.join(base_path, organism, 'tfrecords', f'{subset}-*.tfr')
    
    # Use glob to find all files matching the pattern
    files = glob.glob(search_pattern)
    
    # Sort the values by the integer index at the end: {subset}-{index}.tfr
    # Example: 'data/human/tfrecords/train-12.tfr' -> '12' -> 12
    return sorted(files, key=lambda x: int(os.path.basename(x).split('-')[-1].split('.')[0]))

def deserialize(serialized_example, metadata):
  """Deserialize bytes stored in TFRecordFile."""
  feature_map = {
      'sequence': tf.io.FixedLenFeature([], tf.string), # Raw DNA sequence
      'target': tf.io.FixedLenFeature([], tf.string), # Experimental measurements (e.g. CAGE and ChIP-seq signals)
  }
  # Parse the input tf.Example proto using the dictionary above. 
  example = tf.io.parse_example(serialized_example, feature_map)
  # Decode and reshape the sequence
  sequence = tf.io.decode_raw(example['sequence'], tf.bool)
  sequence = tf.reshape(sequence, (metadata['seq_length'], 4)) # Reshape to (seq_length, 4)
  sequence = tf.cast(sequence, tf.float32) # Convert bool to float32
  # Decode and reshape the target
  target = tf.io.decode_raw(example['target'], tf.float16)
  target = tf.reshape(target, # Reshape to (target_length, num_targets)
                      (metadata['target_length'], metadata['num_targets']))
  target = tf.cast(target, tf.float32) # Convert float16 to float32

  return {'sequence': sequence,
          'target': target}


## Load dataset

In [9]:
df_targets_human = get_targets('human')
df_targets_human.head()

,index,genome,identifier,file,clip,scale,sum_stat,description
0,0,0,ENCFF833POA,/home/drk/tillage/datasets/human/dnase/encode/...,32,2,mean,DNASE:cerebellum male adult (27 years) and mal...
1,1,0,ENCFF110QGM,/home/drk/tillage/datasets/human/dnase/encode/...,32,2,mean,DNASE:frontal cortex male adult (27 years) and...
2,2,0,ENCFF880MKD,/home/drk/tillage/datasets/human/dnase/encode/...,32,2,mean,DNASE:chorion
3,3,0,ENCFF463ZLQ,/home/drk/tillage/datasets/human/dnase/encode/...,32,2,mean,DNASE:Ishikawa treated with 0.02% dimethyl sul...
4,4,0,ENCFF890OGQ,/home/drk/tillage/datasets/human/dnase/encode/...,32,2,mean,DNASE:GM03348


In [10]:
human_dataset = get_dataset('human', 'train').batch(1).repeat()
mouse_dataset = get_dataset('mouse', 'train').batch(1).repeat()
human_mouse_dataset = tf.data.Dataset.zip((human_dataset, mouse_dataset)).prefetch(2)

2026-01-03 18:45:58.843844: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-03 18:45:58.846582: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:923] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-03 18:45:58.846620: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1733] Found device 0 with properties: 
pciBusID: 0000:01:00.0 name: NVIDIA RTX 4500 Ada Generation computeCapability: 8.9
coreClock: 2.58GHz coreCount: 60 deviceMemorySize: 23.99GiB deviceMemoryBandwidth: 402.38GiB/s
2026-01-03 18:45:58.846793: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:923] could not open file to read NUMA node: /sys/bus/p

In [11]:
it = iter(mouse_dataset)
example = next(it)

2026-01-03 18:46:04.970825: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:176] None of the MLIR Optimization Passes are enabled (registered 2)
2026-01-03 18:46:04.971163: I tensorflow/core/platform/profile_utils/cpu_utils.cc:114] CPU Frequency: 1996795000 Hz


In [12]:
# Example input
it = iter(human_mouse_dataset)
example = next(it)
for i in range(len(example)):
  print(['human', 'mouse'][i])
  print({k: (v.shape, v.dtype) for k,v in example[i].items()})

human
{'sequence': (TensorShape([1, 131072, 4]), tf.float32), 'target': (TensorShape([1, 896, 5313]), tf.float32)}
mouse
{'sequence': (TensorShape([1, 131072, 4]), tf.float32), 'target': (TensorShape([1, 896, 1643]), tf.float32)}


## Model training

In [ ]:
def create_step_function(model, optimizer):

  @tf.function
  def train_step(batch, head, optimizer_clip_norm_global=0.2):
    with tf.GradientTape() as tape:
      outputs = model(batch['sequence'], is_training=True)[head] # Forward pass
      loss = tf.reduce_mean( # Compute Poisson loss
          tf.keras.losses.poisson(batch['target'], outputs))

    gradients = tape.gradient(loss, model.trainable_variables) # Compute gradients
    optimizer.apply(gradients, model.trainable_variables) # Apply gradients

    return loss
  return train_step

In [15]:
learning_rate = tf.Variable(0., trainable=False, name='learning_rate')
optimizer = snt.optimizers.Adam(learning_rate=learning_rate)
num_warmup_steps = 5000
target_learning_rate = 0.0005

model = enformer.Enformer(channels=1536 // 4,  # Use 4x fewer channels to train faster.
                          num_heads=8,
                          num_transformer_layers=11,
                          pooling_type='max')

train_step = create_step_function(model, optimizer)

In [ ]:
# Train the model
steps_per_epoch = 20
num_epochs = 5

data_it = iter(human_mouse_dataset)
global_step = 0
for epoch_i in range(num_epochs):
  for i in tqdm(range(steps_per_epoch)):
    global_step += 1

    if global_step > 1:
      # Warmup learning rate
      learning_rate_frac = tf.math.minimum( # Calculate fraction for warmup according to steps
                                            1.0, 
                                            global_step / tf.math.maximum(1.0, num_warmup_steps) # Avoid division by zero. The denominator must be non-zero. 
                                          )      
      learning_rate.assign(target_learning_rate * learning_rate_frac)

    batch_human, batch_mouse = next(data_it)

    loss_human = train_step(batch=batch_human, head='human')
    loss_mouse = train_step(batch=batch_mouse, head='mouse')

  # End of epoch.
  print('')
  print('loss_human', loss_human.numpy(),
        'loss_mouse', loss_mouse.numpy(),
        'learning_rate', optimizer.learning_rate.numpy()
        )

## Evaluate

In [ ]:
# @title `PearsonR` and `R2` metrics

def _reduced_shape(shape, axis):
  if axis is None:
    return tf.TensorShape([])
  return tf.TensorShape([d for i, d in enumerate(shape) if i not in axis])


class CorrelationStats(tf.keras.metrics.Metric):
  """Contains shared code for PearsonR and R2."""

  def __init__(self, reduce_axis=None, name='pearsonr'):
    """Pearson correlation coefficient.

    Args:
      reduce_axis: Specifies over which axis to compute the correlation (say
        (0, 1). If not specified, it will compute the correlation across the
        whole tensor.
      name: Metric name.
    """
    super(CorrelationStats, self).__init__(name=name)
    self._reduce_axis = reduce_axis
    self._shape = None  # Specified in _initialize.

  def _initialize(self, input_shape):
    # Remaining dimensions after reducing over self._reduce_axis.
    self._shape = _reduced_shape(input_shape, self._reduce_axis)

    weight_kwargs = dict(shape=self._shape, initializer='zeros')
    self._count = self.add_weight(name='count', **weight_kwargs)
    self._product_sum = self.add_weight(name='product_sum', **weight_kwargs)
    self._true_sum = self.add_weight(name='true_sum', **weight_kwargs)
    self._true_squared_sum = self.add_weight(name='true_squared_sum',
                                             **weight_kwargs)
    self._pred_sum = self.add_weight(name='pred_sum', **weight_kwargs)
    self._pred_squared_sum = self.add_weight(name='pred_squared_sum',
                                             **weight_kwargs)

  def update_state(self, y_true, y_pred, sample_weight=None):
    """Update the metric state.

    Args:
      y_true: Multi-dimensional float tensor [batch, ...] containing the ground
        truth values.
      y_pred: float tensor with the same shape as y_true containing predicted
        values.
      sample_weight: 1D tensor aligned with y_true batch dimension specifying
        the weight of individual observations.
    """
    if self._shape is None:
      # Explicit initialization check.
      self._initialize(y_true.shape)
    y_true.shape.assert_is_compatible_with(y_pred.shape)
    y_true = tf.cast(y_true, 'float32')
    y_pred = tf.cast(y_pred, 'float32')

    self._product_sum.assign_add(
        tf.reduce_sum(y_true * y_pred, axis=self._reduce_axis))

    self._true_sum.assign_add(
        tf.reduce_sum(y_true, axis=self._reduce_axis))

    self._true_squared_sum.assign_add(
        tf.reduce_sum(tf.math.square(y_true), axis=self._reduce_axis))

    self._pred_sum.assign_add(
        tf.reduce_sum(y_pred, axis=self._reduce_axis))

    self._pred_squared_sum.assign_add(
        tf.reduce_sum(tf.math.square(y_pred), axis=self._reduce_axis))

    self._count.assign_add(
        tf.reduce_sum(tf.ones_like(y_true), axis=self._reduce_axis))

  def result(self):
    raise NotImplementedError('Must be implemented in subclasses.')

  def reset_states(self):
    if self._shape is not None:
      tf.keras.backend.batch_set_value([(v, np.zeros(self._shape))
                                        for v in self.variables])


class PearsonR(CorrelationStats):
  """Pearson correlation coefficient.

  Computed as:
  ((x - x_avg) * (y - y_avg) / sqrt(Var[x] * Var[y])
  """

  def __init__(self, reduce_axis=(0,), name='pearsonr'):
    """Pearson correlation coefficient.

    Args:
      reduce_axis: Specifies over which axis to compute the correlation.
      name: Metric name.
    """
    super(PearsonR, self).__init__(reduce_axis=reduce_axis,
                                   name=name)

  def result(self):
    true_mean = self._true_sum / self._count
    pred_mean = self._pred_sum / self._count

    covariance = (self._product_sum
                  - true_mean * self._pred_sum
                  - pred_mean * self._true_sum
                  + self._count * true_mean * pred_mean)

    true_var = self._true_squared_sum - self._count * tf.math.square(true_mean)
    pred_var = self._pred_squared_sum - self._count * tf.math.square(pred_mean)
    tp_var = tf.math.sqrt(true_var) * tf.math.sqrt(pred_var)
    correlation = covariance / tp_var

    return correlation


class R2(CorrelationStats):
  """R-squared  (fraction of explained variance)."""

  def __init__(self, reduce_axis=None, name='R2'):
    """R-squared metric.

    Args:
      reduce_axis: Specifies over which axis to compute the correlation.
      name: Metric name.
    """
    super(R2, self).__init__(reduce_axis=reduce_axis,
                             name=name)

  def result(self):
    true_mean = self._true_sum / self._count
    total = self._true_squared_sum - self._count * tf.math.square(true_mean)
    residuals = (self._pred_squared_sum - 2 * self._product_sum
                 + self._true_squared_sum)

    return tf.ones_like(residuals) - residuals / total


class MetricDict:
  def __init__(self, metrics):
    self._metrics = metrics

  def update_state(self, y_true, y_pred):
    for k, metric in self._metrics.items():
      metric.update_state(y_true, y_pred)

  def result(self):
    return {k: metric.result() for k, metric in self._metrics.items()}

In [ ]:
def evaluate_model(model, dataset, head, max_steps=None):
  metric = MetricDict({'PearsonR': PearsonR(reduce_axis=(0,1))})
  @tf.function
  def predict(x):
    return model(x, is_training=False)[head]

  for i, batch in tqdm(enumerate(dataset)):
    if max_steps is not None and i > max_steps:
      break
    metric.update_state(batch['target'], predict(batch['sequence']))

  return metric.result()

In [ ]:
metrics_human = evaluate_model(model,
                               dataset=get_dataset('human', 'valid').batch(1).prefetch(2),
                               head='human',
                               max_steps=100)
print('')
print({k: v.numpy().mean() for k, v in metrics_human.items()})

101it [00:23,  6.27it/s]


{'PearsonR': 0.0028573992}


In [ ]:
metrics_mouse = evaluate_model(model,
                               dataset=get_dataset('mouse', 'valid').batch(1).prefetch(2),
                               head='mouse',
                               max_steps=100)
print('')
print({k: v.numpy().mean() for k, v in metrics_mouse.items()})

101it [00:21,  6.54it/s]


{'PearsonR': 0.005183698}


# Restore Checkpoint

Note: For the TF-Hub Enformer model, the required input sequence length is 393,216 which actually gets cropped within the model to 196,608. The open source module does not internally crop the sequence. Therefore, the code below crops the central `196,608 bp` of the longer sequence to reproduce the output of the TF hub from the reloaded checkpoint.

In [ ]:
np.random.seed(42)
EXTENDED_SEQ_LENGTH = 393_216
SEQ_LENGTH = 196_608
inputs = np.array(np.random.random((1, EXTENDED_SEQ_LENGTH, 4)), dtype=np.float32)
inputs_cropped = enformer.TargetLengthCrop1D(SEQ_LENGTH)(inputs)

In [ ]:
checkpoint_gs_path = 'gs://dm-enformer/models/enformer/sonnet_weights/*'
checkpoint_path = '/tmp/enformer_checkpoint'

In [ ]:
!mkdir /tmp/enformer_checkpoint

mkdir: cannot create directory ‘/tmp/enformer_checkpoint’: File exists


In [ ]:
# Copy checkpoints from GCS to temporary directory.
# This will take a while as the checkpoint is ~ 1GB.
for file_path in tf.io.gfile.glob(checkpoint_gs_path):
  print(file_path)
  file_name = os.path.basename(file_path)
  tf.io.gfile.copy(file_path, f'{checkpoint_path}/{file_name}', overwrite=True)

gs://dm-enformer/models/enformer/sonnet_weights/checkpoint
gs://dm-enformer/models/enformer/sonnet_weights/enformer-fine-tuned-human-1.data-00000-of-00001
gs://dm-enformer/models/enformer/sonnet_weights/enformer-fine-tuned-human-1.index


In [ ]:
!ls -lh /tmp/enformer_checkpoint

total 959M
-rw-r--r-- 1 root root  111 May 25 10:58 checkpoint
-rw-r--r-- 1 root root 959M May 25 10:59 enformer-fine-tuned-human-1.data-00000-of-00001
-rw-r--r-- 1 root root 5.7K May 25 10:59 enformer-fine-tuned-human-1.index


In [ ]:
enformer_model = enformer.Enformer()

In [ ]:
checkpoint = tf.train.Checkpoint(module=enformer_model)

In [ ]:
latest = tf.train.latest_checkpoint(checkpoint_path)
print(latest)
status = checkpoint.restore(latest)

/tmp/enformer_checkpoint/enformer-fine-tuned-human-1


In [ ]:
# Using `is_training=False` to match TF-hub predict_on_batch function.
restored_predictions = enformer_model(inputs_cropped, is_training=False)

In [ ]:
import tensorflow_hub as hub
enformer_tf_hub_model = hub.load("https://tfhub.dev/deepmind/enformer/1").model

In [ ]:
hub_predictions = enformer_tf_hub_model.predict_on_batch(inputs)

In [ ]:
np.allclose(hub_predictions['human'], restored_predictions['human'], atol=1e-5)

True

In [ ]:
# Can run with 'is_training=True' but note that this will
# change the predictions as the batch statistics will be updated
# and the outputs will likley not match the TF-hub model.
# enformer(inputs_cropped, is_training=True)